# Оценка качества моделей рекомендаций

## Описание
Данный ноутбук выполняет:
1. Загрузку обученных эмбеддингов ALS и Neural BPR
2. Нормализацию эмбеддингов для косинусного сходства
3. Вычисление метрик Precision@10, Recall@10, F1@10
4. Сравнение с baseline (Popular)
5. Сохранение результатов в JSON

## Функции оценки

```python
def normalize(x: np.ndarray) -> np.ndarray:
    """
    Нормализует векторы эмбеддингов по L2-норме.
    
    Args:
        x: матрица эмбеддингов [n_samples x dim]
        
    Returns:
        np.ndarray: нормализованная матрица
    """

def evaluate(emb_user: np.ndarray, emb_item: np.ndarray, name: str) -> tuple:
    """
    Вычисляет метрики Precision@K, Recall@K, F1@K.
    
    Args:
        emb_user: матрица эмбеддингов пользователей [n_users x dim]
        emb_item: матрица эмбеддингов товаров [n_items x dim]
        name: название модели для вывода
        
    Returns:
        tuple: (precision, recall, f1)
    """

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
%cd /content/drive/MyDrive/VKR_TECD/

/content/drive/MyDrive/VKR_TECD


In [ ]:
import numpy as np
from scipy.sparse import load_npz
import json

print("ШАГ 4: ОЦЕНКА КАЧЕСТВА ВСЕХ МОДЕЛЕЙ")

R = load_npz("output/train_matrix.npz")

# Загружаем эмбеддинги ALS
user_als = np.load("output/user_factors_als.npy")
item_als = np.load("output/item_factors_als.npy")

# Загружаем эмбеддинги Neural BPR
user_neural = np.load("output/user_embeddings_neural.npy")
item_neural = np.load("output/item_embeddings_neural.npy")

def normalize(x):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1
    return x / norms

user_als_norm = normalize(user_als)
item_als_norm = normalize(item_als)
user_neural_norm = normalize(user_neural)
item_neural_norm = normalize(item_neural)

K = 10
TEST_USERS = 500

np.random.seed(42)
test_users = np.random.choice(R.shape[0], size=min(TEST_USERS, R.shape[0]), replace=False)

item_pop = np.array(R.sum(axis=0)).flatten()
popular_items = np.argsort(item_pop)[::-1][:K]

def evaluate(emb_user, emb_item, name):
    precisions, recalls = [], []
    for u in test_users:
        true_items = set(R[u].indices)
        if len(true_items) == 0:
            continue
        scores = np.dot(emb_item, emb_user[u])
        scores[list(true_items)] = -np.inf
        top = np.argsort(scores)[::-1][:K]
        hits = len(set(top) & true_items)
        precisions.append(hits / K)
        recalls.append(hits / len(true_items))
    p = np.mean(precisions)
    r = np.mean(recalls)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    print(f"  {name}: Precision@{K}={p:.4f}, Recall@{K}={r:.4f}, F1@{K}={f1:.4f}")
    return p, r, f1

print("Оценка Popular...")
prec_base, rec_base = [], []
for u in test_users:
    true_items = set(R[u].indices)
    if len(true_items) == 0:
        continue
    hits = len(set(popular_items) & true_items)
    prec_base.append(hits / K)
    rec_base.append(hits / len(true_items))

p_base = np.mean(prec_base)
r_base = np.mean(rec_base)
f1_base = 2 * p_base * r_base / (p_base + r_base)
print(f"  Popular: Precision@{K}={p_base:.4f}, Recall@{K}={r_base:.4f}, F1@{K}={f1_base:.4f}")

print("\nОценка ALS...")
p_als, r_als, f1_als = evaluate(user_als_norm, item_als_norm, "ALS")

print("\nОценка Neural BPR...")
p_neural, r_neural, f1_neural = evaluate(user_neural_norm, item_neural_norm, "Neural BPR")

print("\n" + "=" * 60)
print("ИТОГОВАЯ ТАБЛИЦА")
print("=" * 60)
print(f"{'Модель':<15} {'Precision@10':<15} {'Recall@10':<15} {'F1@10':<15}")
print("-" * 60)
print(f"{'Popular':<15} {p_base:<15.4f} {r_base:<15.4f} {f1_base:<15.4f}")
print(f"{'ALS':<15} {p_als:<15.4f} {r_als:<15.4f} {f1_als:<15.4f}")
print(f"{'Neural BPR':<15} {p_neural:<15.4f} {r_neural:<15.4f} {f1_neural:<15.4f}")
print("=" * 60)

results = {
    'Popular': {'Precision@10': p_base, 'Recall@10': r_base, 'F1@10': f1_base},
    'ALS': {'Precision@10': p_als, 'Recall@10': r_als, 'F1@10': f1_als},
    'Neural_BPR': {'Precision@10': p_neural, 'Recall@10': r_neural, 'F1@10': f1_neural}
}
with open("output/all_results_final.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nРезультаты сохранены в output/all_results_final.json")

ШАГ 4: ОЦЕНКА КАЧЕСТВА ВСЕХ МОДЕЛЕЙ
Оценка Popular...
  Popular: Precision@10=0.2522, Recall@10=0.0723, F1@10=0.1124

Оценка ALS...
  ALS: Precision@10=0.0000, Recall@10=0.0000, F1@10=0.0000

Оценка Neural BPR...
  Neural BPR: Precision@10=0.0000, Recall@10=0.0000, F1@10=0.0000

ИТОГОВАЯ ТАБЛИЦА
Модель          Precision@10    Recall@10       F1@10          
------------------------------------------------------------
Popular         0.2522          0.0723          0.1124         
ALS             0.0000          0.0000          0.0000         
Neural BPR      0.0000          0.0000          0.0000         

Результаты сохранены в output/all_results_final.json
